In [8]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


## Environment Setup, Secrets & W&B Initialization

In [9]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import LabelEncoder
import wandb
from kaggle_secrets import UserSecretsClient

# Set random seed for reproducibility
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    
seed_everything(seed=42)

# Fetch W&B API key securely from Kaggle secrets
user_secrets = UserSecretsClient()
wandb_api_key = user_secrets.get_secret("WanDB_API_KEY")

# Log in to W&B using the secret key
wandb.login(key=wandb_api_key)

# W&B Init 
wandb.init(
    project="DL-23f1000967-notebook-t22026",
    name="model1_tfidf_scratch",
    config={
        "model_type": "FROM_scratch_CLASSIFIER",
        "feature_extractor": "TFIDF",
        "tfidf_features": 2500, # Dynamically maps to max_features below
        "hidden_dim": 128,
        "dropout": 0.3,
        "learning_rate": 0.005,
        "batch_size": 64,
        "epochs": 40
    }
)

# Access config and device
config = wandb.config
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


Using device: cuda


## Load Competition Data

In [10]:
train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
print(f"Train Dataset Shape: {train_df.shape}")
print(f"Test Dataset Shape:  {test_df.shape}\n")

Train Dataset Shape: (2000, 8)
Test Dataset Shape:  (500, 7)



## Text Feature Engineering Engine (Reading from config)

In [11]:
def extract_hybrid_features(df, tfidf_vectorizer=None, is_train=True):
    text_corpus = df['prompt'].fillna("").tolist()
    for option_col in ['A', 'B', 'C', 'D', 'E']:
        text_corpus += df[option_col].fillna("").tolist()
        
    if is_train:
        # Reading max_features dynamically from wandb config
        tfidf_vectorizer = TfidfVectorizer(
            stop_words='english', 
            max_features=config.tfidf_features, 
            sublinear_tf=True
        )
        tfidf_vectorizer.fit(text_corpus)
        
    engineered_features = []
    
    for idx, row in df.iterrows():
        p_text = str(row['prompt'])
        options = [str(row['A']), str(row['B']), str(row['C']), str(row['D']), str(row['E'])]
        
        p_vec = tfidf_vectorizer.transform([p_text])
        opt_vecs = tfidf_vectorizer.transform(options)
        
        cos_similarities = cosine_similarity(p_vec, opt_vecs).flatten() 
        
        p_len = max(len(p_text), 1)
        length_ratios = [len(opt) / p_len for opt in options]
        
        combined_row = np.concatenate([cos_similarities, length_ratios])
        engineered_features.append(combined_row)
        
    return np.array(engineered_features), tfidf_vectorizer

# Run processing
X_train_raw, vectorizer = extract_hybrid_features(train_df, is_train=True)
X_test_raw, _ = extract_hybrid_features(test_df, tfidf_vectorizer=vectorizer, is_train=False)

encoder = LabelEncoder()
y_train_raw = encoder.fit_transform(train_df['answer'])

print(f"Engineered Matrix (Train): {X_train_raw.shape}")

Engineered Matrix (Train): (2000, 10)


## Prepare Dataloaders

In [12]:
X_train_tensor = torch.tensor(X_train_raw, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_raw, dtype=torch.long)
X_test_tensor = torch.tensor(X_test_raw, dtype=torch.float32)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)

## Forward Neural Network Architecture

In [13]:
class MCQForwardNN(nn.Module):
    def __init__(self, input_features):
        super(MCQForwardNN, self).__init__()
        
        self.network = nn.Sequential(
            nn.Linear(input_features, config.hidden_dim),
            nn.BatchNorm1d(config.hidden_dim),
            nn.ReLU(),
            nn.Dropout(config.dropout),
            
            nn.Linear(config.hidden_dim, config.hidden_dim // 2),
            nn.BatchNorm1d(config.hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(config.dropout),
            
            nn.Linear(config.hidden_dim // 2, 5) 
        )
        
    def forward(self, x):
        return self.network(x)

input_dim = X_train_raw.shape[1]
model = MCQForwardNN(input_features=input_dim).to(device)

## Training Loop with

In [14]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=0.01)

print("Training and Logging via W&B")

for epoch in range(1, config.epochs + 1):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        logits = model(batch_x)
        loss = criterion(logits, batch_y)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * batch_x.size(0)
        
        # Track accuracy metric
        _, predicted = torch.max(logits, 1)
        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()
        
    epoch_loss = running_loss / len(train_loader.dataset)
    epoch_acc = correct / total
    
    # Send metrics straight to W&B dashboard
    wandb.log({
        "epoch": epoch,
        "loss": epoch_loss,
        "accuracy": epoch_acc,
        "lr": optimizer.param_groups[0]['lr']
    })
    
    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch [{epoch:02d}/{config.epochs}] | Loss: {epoch_loss:.5f} | Acc: {epoch_acc:.4f}")

# Close the W&B Run properly
wandb.finish()
print("Training cycle complete & W&B updated successfully!")

--- Training and Logging via W&B ---
Epoch [01/40] | Loss: 1.45953 | Acc: 0.3970
Epoch [05/40] | Loss: 1.20923 | Acc: 0.4810
Epoch [10/40] | Loss: 1.10049 | Acc: 0.5450
Epoch [15/40] | Loss: 1.02769 | Acc: 0.5710
Epoch [20/40] | Loss: 0.98984 | Acc: 0.5900
Epoch [25/40] | Loss: 0.95245 | Acc: 0.6215
Epoch [30/40] | Loss: 0.91308 | Acc: 0.6205
Epoch [35/40] | Loss: 0.89961 | Acc: 0.6340
Epoch [40/40] | Loss: 0.86802 | Acc: 0.6630


accuracy,▁▃▃▃▃▄▄▄▄▅▅▆▅▅▆▆▆▆▆▆▆▇▇▆▇▇▇▆▆▇▇▇██▇█████
epoch,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
loss,█▆▆▅▅▅▄▄▄▄▄▃▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▁▁▂▁▁▁▁▁
lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
accuracy,0.663
epoch,40
loss,0.86802
lr,0.005


Training cycle complete & W&B updated successfully!


## Rank Inference & Submission Export

In [15]:
model.eval()
with torch.no_grad():
    test_inputs = X_test_tensor.to(device)
    raw_logits = model(test_inputs)
    probabilities = torch.softmax(raw_logits, dim=1).cpu().numpy()

classes = np.array(['A', 'B', 'C', 'D', 'E'])
map_predictions = []

for row_probs in probabilities:
    top_3_indices = np.argsort(row_probs)[::-1][:3]
    top_3_labels = classes[top_3_indices]
    map_predictions.append(" ".join(top_3_labels))

submission_df = pd.DataFrame({
    'id': test_df['id'],
    'prediction': map_predictions
})
submission_df.to_csv('submission.csv', index=False)
display(submission_df.head(5))

,id,prediction
0,1,B D C
1,2,B C D
2,3,B C E
3,4,E D A
4,5,B E A
